In [0]:
from pyspark.sql.functions import col, when, trim, regexp_replace, concat_ws, split
from pyspark.sql import functions as F

In [0]:
df = spark.table("novacart_catalog.001_bronze.exchange_rates")

df.display()

In [0]:
for col_name in df.columns:
    new_col_name = col_name[0].upper() + col_name[1:] if len(col_name) > 0 else col_name
    df = df.withColumnRenamed(col_name, new_col_name)

df.display()

In [0]:
df = df.withColumn(
    "Effective_date",
    F.coalesce(
        F.expr("try_to_date(Effective_date, 'dd-MM-yyyy')"),
        F.expr("try_to_date(Effective_date, 'd/M/yyyy')"),
        F.expr("try_to_date(Effective_date, 'M/d/yyyy')"),
        F.expr("try_to_date(Effective_date, 'MM-dd-yyyy')"),
    )
)
display(df.limit(5))

In [0]:
df.write.format("delta") .mode("overwrite") .option("overwriteSchema", "true") .saveAsTable("novacart_catalog.002_silver.exchange_rates")